# 09 · Explain H₂ energy / မော်လီကျူး စွမ်းအင်ကို လေ့လာခြင်း

**Unreleased source-checkout lesson.** Install this checkout with the tutorial
extra. This fixed-geometry lesson does not need PySCF. To regenerate integrals
and compare a bond-length curve against independent PySCF FCI, install
`python -m pip install -e '.[chemistry,tutorial]'` and run
`python examples/h2_vqe.py`.

The coefficients below were generated with PySCF 2.14.0, RHF/STO-3G, neutral
singlet H₂ at 0.735 Å. They use Jordan–Wigner mapping and interleaved alpha/beta
spin orbitals in q0-first order. Total energies include nuclear repulsion.
The saved classical FCI reference is -1.1373060357534004 Hartree.

H₂ ကိုသာ အခြေခံထားသော ဥပမာဖြစ်ပြီး molecule အကြီးများ သို့မဟုတ် ဆေးဝါး၏
ထိရောက်မှုကို တွက်ချက်ထားခြင်း မဟုတ်ပါ။


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nexqus
from nexqus import Circuit, Simulator, Parameter, Noise, NoiseModel, algorithms
assert nexqus.__version__ == '1.2.0', 'Install NexQus SDK 1.2.0 in this kernel environment and restart it'
print('NexQus SDK:', nexqus.__version__)


In [ ]:
from nexqus import PauliHamiltonian, Adam
from nexqus.chemistry import h2_pair_ansatz
terms = [
    (-.09057898608834797, 'IIII'), (-.22575349222402394, 'IIIZ'),
    (-.22575349222402394, 'IIZI'), (.17464343068300442, 'IIZZ'),
    (.1721839326191555, 'IZII'), (.1209126326177663, 'IZIZ'),
    (.16614543256382414, 'IZZI'), (-.04523279994605785, 'XXYY'),
    (.04523279994605785, 'XYYX'), (.04523279994605785, 'YXXY'),
    (-.04523279994605785, 'YYXX'), (.1721839326191555, 'ZIII'),
    (.16614543256382414, 'ZIIZ'), (.1209126326177663, 'ZIZI'),
    (.16892753870087907, 'ZZII')]
hamiltonian = PauliHamiltonian(4, terms)
ansatz = h2_pair_ansatz(Parameter('theta'))
energy = hamiltonian.function(ansatz)
parameters = {'theta': 0.}; optimizer = Adam(.04); energies = []
hf_energy = energy(parameters)
for epoch in range(180):
    value, gradient = energy.value_and_grad(parameters)
    energies.append(value)
    parameters = optimizer.step(parameters, gradient)
vqe_energy = energy(parameters)
fci_energy = -1.1373060357534004
print('Classical Hartree–Fock:', hf_energy)
print('Classical FCI reference:', fci_energy)
print('NexQus VQE:', vqe_energy)
assert abs(hf_energy - (-1.116998996754004)) < 1e-8
assert abs(vqe_energy-fci_energy) < 2e-7


## What is learned, and where does the energy come from?

The ansatz output is `cos(theta/2)|1100> + sin(theta/2)|0011>`.
It mixes a doubly occupied bonding orbital with a doubly excited determinant.
Every final state contains two electrons. This is a specialized H₂ family,
not a general-purpose chemistry ansatz.

The gate decomposition can temporarily leave the two-electron sector.
Intermediate Bloch states are circuit internals, not a molecular reaction
path. The explorer's `ZZII` readout is a single Pauli observable, not the
whole molecular energy. The plot below explicitly sums all energy terms.


In [ ]:
result = Simulator().run(ansatz.bind(parameters), shots=0, retain_state=True)
contributions = [c*result.expectation(word) for c, word in hamiltonian.terms]
assert abs(sum(contributions)-vqe_energy) < 1e-10
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout='constrained')
axes[0].plot(energies, label='NexQus VQE')
axes[0].axhline(fci_energy, linestyle='--', label='Classical FCI')
axes[0].set(xlabel='Update', ylabel='Total energy / Hartree'); axes[0].legend()
axes[1].barh([word for _, word in hamiltonian.terms], contributions)
axes[1].set_xlabel('Energy contribution / Hartree')
plt.show()
from IPython.display import display
display(ansatz.explain(bindings=parameters, observable='ZZII',
    parameter_roles={'theta': 'weight'}, title='H₂ · determinant mixing'))
